<a href="https://colab.research.google.com/github/IKKNIGHT/Protein_interactions/blob/main/Phase9_Cross_Target_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: cross-target analysis of the de novo PPI benchmark
Reads every per-target `gate1_results.csv`-style file the sweep wrote (one combined `results/sweep/*_results.csv` per target, or the single `gate1/gate1_results.csv`), and answers the questions the benchmark exists for:
1. Per target x arm: pose self-consistency, interface F1 vs native, ipTM, interface PAE.
2. **Does the GNN beat the SASA x hydrophobicity baseline?** Paired across targets (Wilcoxon), on pose self-consistency and on best-F1.
3. How close is GNN to the ORACLE ceiling, and how far above SCRAMBLED.
4. Which targets the pipeline simply cannot do (all arms fail), separated from where the GNN specifically wins.

This is read-only analysis: no GPU. Point `RESULTS_GLOB` at wherever the sweep saved results.

In [1]:
# ── Cell 1: Load every target's results ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json
import numpy as np, pandas as pd
BASE = '/content/drive/MyDrive/Protein_interactions'
# the sweep eval notebook writes one combined CSV; also accept per-target files if you split runs
CANDS = [f'{BASE}/gate1/gate1_results.csv', *glob.glob(f'{BASE}/**/*gate*results*.csv', recursive=True),
         *glob.glob(f'{BASE}/results/**/*results*.csv', recursive=True)]
paths = sorted(set(p for p in CANDS if os.path.exists(p)))
print('result files found:'); [print('  ', p) for p in paths]
frames = []
for p in paths:
    d = pd.read_csv(p); d['_src'] = os.path.basename(p); frames.append(d)
assert frames, 'no results CSVs found - run the sweep eval notebook first'
R = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['run_id'], keep='last')
# target label: prefer explicit column, else parse from run_id suffix (..__<id>_<TARGET> or ..__<TARGET>)
def tgt_of(row):
    if 'spec' in row and isinstance(row['spec'], str) and '_' in row['spec']:
        return row['spec'].split('_', 2)[-1].rsplit('_', 1)[0] if row['spec'].count('_')>=2 else row['spec']
    rid = str(row['run_id']); tail = rid.split('__')[-1]
    return tail.split('_',1)[-1] if '_' in tail and tail.split('_',1)[0].isdigit() else tail
R['target'] = R['target'] if 'target' in R and R['target'].notna().all() else R.apply(tgt_of, axis=1)
print(f'\n{len(R)} runs across {R.target.nunique()} targets; kinds:', R.kind.value_counts().to_dict())

Mounted at /content/drive
result files found:
   /content/drive/MyDrive/Protein_interactions/gate1/gate1_results.csv

1280 runs across 20 targets; kinds: {'design': 1200, 'shuffled': 60, 'native': 20}


In [2]:
# ── Cell 2: Per-target x arm summary table ─────────────────────────────────────
import sys; sys.path.insert(0, BASE)  # pipeline_metrics.py lives at the repo root
from pipeline_metrics import POSE_RMSD_MAX  # was a separate local POSE_OK=8.0 -- same quantity as Phase9 cells 7/8, now single-sourced
des = R[(R.kind == 'design') & R.pose_rmsd.notna()].copy()
def arm_stats(g):
    return pd.Series({'n': len(g), 'median_iptm': g.iptm.median(), 'median_pae': g.iface_pae.median(),
                      'best_f1': g.f1.max(), 'median_f1': g.f1.median(),
                      'frac_self_consistent': (g.pose_rmsd <= POSE_RMSD_MAX).mean(), 'median_pose_rmsd': g.pose_rmsd.median()})
per = des.groupby(['target', 'arm']).apply(arm_stats).reset_index()
pivot_sc = per.pivot(index='target', columns='arm', values='frac_self_consistent')
pivot_f1 = per.pivot(index='target', columns='arm', values='best_f1')
pd.set_option('display.width', 200)
print('POSE SELF-CONSISTENCY (fraction of designs Boltz re-places within 8 A of the RFdiffusion pose):')
print(pivot_sc.round(2).to_string())
print('\nBEST INTERFACE F1 vs native (per arm):')
print(pivot_f1.round(2).to_string())
per.to_csv(f'{BASE}/results/phase9_per_target_arm.csv', index=False)

POSE SELF-CONSISTENCY (fraction of designs Boltz re-places within 8 A of the RFdiffusion pose):
arm              baseline   gnn  oracle  scrambled
target                                            
ARF1                 0.20  0.87    0.60       0.13
BCL2                 0.13  0.87    0.80       0.13
BCL2L1 (Bcl-xL)      0.60  0.73    0.80       0.00
BRCA1                0.33  0.80    0.87       0.40
CDK2                 0.13  0.13    0.13       0.00
CREBBP (KIX)         0.47  0.47    0.47       0.07
CTNNB1               0.00  0.00    0.53       0.40
FAS                  0.07  0.07    0.00       0.00
GNAI1                0.27  0.33    0.40       0.00
HRAS                 0.00  0.00    0.53       0.07
JUN                  0.87  1.00    0.87       0.33
KEAP1                0.00  0.00    0.27       0.07
MDM2                 0.53  0.87    0.87       0.33
MEN1                 0.00  0.27    0.00       0.00
MYC                  0.80  0.80    0.87       0.53
PCNA                 0.13  0.47    0.

/tmp/ipykernel_878/800889582.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per = des.groupby(['target', 'arm']).apply(arm_stats).reset_index()


In [3]:
# ── Cell 3: Does the GNN beat the baseline? (paired across targets) ────────────
from scipy.stats import wilcoxon
def paired(metric_pivot, a, b, label):
    d = metric_pivot[[a, b]].dropna()
    if len(d) < 3: print(f'{label}: too few paired targets ({len(d)})'); return
    wins = int((d[a] > d[b]).sum()); ties = int((d[a] == d[b]).sum())
    try: p = wilcoxon(d[a], d[b], zero_method='wilcox').pvalue
    except ValueError: p = float('nan')
    print(f'{label}: {a} > {b} on {wins}/{len(d)} targets (ties {ties}); '
          f'median {a}={d[a].median():.2f} vs {b}={d[b].median():.2f}; Wilcoxon p={p:.3g}')

print('=== Pose self-consistency ===')
for a, b in [('gnn','baseline'), ('gnn','scrambled'), ('oracle','gnn'), ('oracle','scrambled')]:
    paired(pivot_sc, a, b, f'{a} vs {b}')
print('\n=== Best interface F1 ===')
for a, b in [('gnn','baseline'), ('gnn','scrambled'), ('oracle','gnn')]:
    paired(pivot_f1, a, b, f'{a} vs {b}')

=== Pose self-consistency ===
gnn vs baseline: gnn > baseline on 13/20 targets (ties 7); median gnn=0.47 vs baseline=0.20; Wilcoxon p=0.00142
gnn vs scrambled: gnn > scrambled on 17/20 targets (ties 0); median gnn=0.47 vs scrambled=0.07; Wilcoxon p=0.000769
oracle vs gnn: oracle > gnn on 7/20 targets (ties 3); median oracle=0.50 vs gnn=0.47; Wilcoxon p=0.394
oracle vs scrambled: oracle > scrambled on 17/20 targets (ties 2); median oracle=0.50 vs scrambled=0.07; Wilcoxon p=0.00038

=== Best interface F1 ===
gnn vs baseline: gnn > baseline on 13/20 targets (ties 0); median gnn=0.75 vs baseline=0.73; Wilcoxon p=0.452
gnn vs scrambled: gnn > scrambled on 15/20 targets (ties 0); median gnn=0.75 vs scrambled=0.78; Wilcoxon p=0.277
oracle vs gnn: oracle > gnn on 12/20 targets (ties 1); median oracle=0.81 vs gnn=0.75; Wilcoxon p=0.227


In [4]:
# ── Cell 4: Where does the pipeline work, and where does the GNN specifically win? ──
SC = 0.5   # call an arm "successful" on a target if >=50% of its designs are pose-self-consistent
tab = pivot_sc.copy()
work_any = tab.max(axis=1) >= SC
gnn_wins = (tab.get('gnn', 0) >= SC) & (tab.get('baseline', 1) < SC)
both = (tab.get('gnn', 0) >= SC) & (tab.get('baseline', 0) >= SC)
dead = tab.max(axis=1) < SC
print(f'Targets where SOME arm works (>= {SC:.0%} self-consistent): {int(work_any.sum())}/{len(tab)}')
print(f'  GNN works AND baseline fails (GNN-specific win): {sorted(tab.index[gnn_wins])}')
print(f'  both GNN and baseline work: {sorted(tab.index[both])}')
print(f'  no arm works (pipeline limit): {sorted(tab.index[dead])}')
# native-control sanity: did Boltz reproduce each native complex?
nat = R[R.kind == 'native'][['target', 'f1', 'iptm']].set_index('target')
print('\nNative-control F1 (low means that target scoring is suspect):')
print(nat.sort_values('f1').round(2).head(10).to_string())

Targets where SOME arm works (>= 50% self-consistent): 11/20
  GNN works AND baseline fails (GNN-specific win): ['ARF1', 'BCL2', 'BRCA1', 'VHL']
  both GNN and baseline work: ['BCL2L1 (Bcl-xL)', 'JUN', 'MDM2', 'MYC', 'VCL']
  no arm works (pipeline limit): ['CDK2', 'CREBBP (KIX)', 'FAS', 'GNAI1', 'KEAP1', 'MEN1', 'PCNA', 'RAD51', 'XIAP (BIR3)']

Native-control F1 (low means that target scoring is suspect):
                   f1  iptm
target                     
FAS              0.46  0.95
XIAP (BIR3)      0.65  0.55
ARF1             0.81  0.91
BRCA1            0.82  0.90
VHL              0.83  0.95
HRAS             0.85  0.69
CDK2             0.89  0.89
CTNNB1           0.89  0.96
BCL2L1 (Bcl-xL)  0.89  0.97
MDM2             0.91  0.96


In [5]:
# ── Cell 5: Headline + save ────────────────────────────────────────────────────
summary = {
    'targets': int(pivot_sc.shape[0]),
    'median_frac_self_consistent': {a: round(float(pivot_sc[a].median()), 3) for a in pivot_sc.columns},
    'median_best_f1': {a: round(float(pivot_f1[a].median()), 3) for a in pivot_f1.columns},
}
print(json.dumps(summary, indent=2))
json.dump(summary, open(f'{BASE}/results/phase9_summary.json', 'w'), indent=2)
print('\nHeadline read:')
if 'gnn' in pivot_sc and 'baseline' in pivot_sc:
    g, b = pivot_sc['gnn'].median(), pivot_sc['baseline'].median()
    print(f'  GNN median self-consistency {g:.2f} vs baseline {b:.2f}'
          + ('  -> GNN adds value' if g > b + 0.05 else '  -> GNN ~ baseline (geometry/hotspots not separating here)'))

{
  "targets": 20,
  "median_frac_self_consistent": {
    "baseline": 0.2,
    "gnn": 0.467,
    "oracle": 0.5,
    "scrambled": 0.067
  },
  "median_best_f1": {
    "baseline": 0.728,
    "gnn": 0.754,
    "oracle": 0.813,
    "scrambled": 0.784
  }
}

Headline read:
  GNN median self-consistency 0.47 vs baseline 0.20  -> GNN adds value


In [6]:
import pandas as pd, numpy as np, sys
from scipy.stats import wilcoxon
sys.path.insert(0, BASE)  # pipeline_metrics.py lives at the repo root
from pipeline_metrics import classify_design
R = pd.read_csv(f'{BASE}/gate1/gate1_results.csv')
trusted = set(pd.read_csv(f'{BASE}/gate1/target_quality.csv', index_col=0).query('trusted').index)
d = R[(R.kind=='design') & R.target.isin(trusted) & R.pose_rmsd.notna()].copy()
# success = Boltz re-places the binder near the RFdiffusion pose AND at the native partner's site
# (see pipeline_metrics.py -- unified with Phase9 cell 8's identical inline formula)
d['hit'], d['self_consistent'] = classify_design(d.pose_rmsd, d.com_dist)
piv = d.groupby(['target','arm']).hit.mean().unstack()
print(piv.round(2).to_string())
print('\nmedian hit-rate:', piv.median().round(3).to_dict())
for a, b in [('gnn','baseline'), ('gnn','scrambled'), ('oracle','gnn')]:
    x = piv[[a, b]].dropna(); w = int((x[a] > x[b]).sum())
    try: p = wilcoxon(x[a], x[b], zero_method='wilcox').pvalue
    except ValueError: p = float('nan')
    print(f'{a} vs {b}: {w}/{len(x)} wins, median {x[a].median():.2f} vs {x[b].median():.2f}, p={p:.3g}')

arm              baseline   gnn  oracle  scrambled
target                                            
ARF1                 0.00  0.87    0.53       0.07
BCL2                 0.07  0.80    0.60       0.00
BCL2L1 (Bcl-xL)      0.53  0.73    0.73       0.00
BRCA1                0.00  0.40    0.60       0.07
CDK2                 0.00  0.00    0.00       0.00
CREBBP (KIX)         0.07  0.00    0.33       0.00
CTNNB1               0.00  0.00    0.00       0.00
GNAI1                0.00  0.00    0.00       0.00
HRAS                 0.00  0.00    0.53       0.00
JUN                  0.33  0.67    0.67       0.20
KEAP1                0.00  0.00    0.20       0.00
MDM2                 0.33  0.87    0.87       0.27
MEN1                 0.00  0.00    0.00       0.00
MYC                  0.20  0.13    0.13       0.13
PCNA                 0.00  0.27    0.13       0.00
RAD51                0.00  0.00    0.07       0.00
VCL                  0.00  0.67    0.47       0.00
VHL                  0.00  0.00

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
#  MASTER DATA EXPORT — run this in Colab (any of the project notebooks, after
#  mounting Drive). Collects every datapoint the project produced into ONE
#  multi-sheet workbook plus flat CSVs, written to the root of Protein_interactions.
#  Safe to re-run; it only reads existing files.
# ══════════════════════════════════════════════════════════════════════════════
import os, re, json, glob, time
import numpy as np, pandas as pd

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass
BASE = '/content/drive/MyDrive/Protein_interactions'
OUT  = f'{BASE}/MASTER'; os.makedirs(OUT, exist_ok=True)
import sys; sys.path.insert(0, BASE)  # pipeline_metrics.py lives at the repo root
from pipeline_metrics import classify_design
rd   = lambda p: pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()
jl   = lambda p: json.load(open(p)) if os.path.exists(p) else {}

# ── 1. design-level results + sequences ───────────────────────────────────────
RES = rd(f'{BASE}/gate1/gate1_results.csv')
SEL = rd(f'{BASE}/gate1/selected_designs.csv')
if not RES.empty and not SEL.empty:
    seqcols = [c for c in ('run_id','seq','binder_len','max_aa_frac','degenerate',
                           'backbone','stem','rank') if c in SEL.columns]
    RES = RES.merge(SEL[seqcols], on='run_id', how='left', suffixes=('', '_sel'))

# ── 1b. write designs.csv + compute designability NOW (bug fix, see note) ────
# BUG (fixed): DESIGNABILITY used to be read in section 4 below, but
# gate1/designability.csv was only ever produced by cell 9, which read
# MASTER/designs.csv -- itself only written by THIS cell's own export loop
# at the very end (section 9). On a first full run, that meant this cell's
# own DESIGNABILITY read happened before its own designs.csv write, so
# DESIGNABILITY silently came out empty and the exported workbook/CSV
# got an empty 'designability' sheet. Inlined cell 9's computation here,
# right after RES is finalized, so the read below (section 4) is fresh.
os.makedirs(f'{BASE}/gate1', exist_ok=True)
if not RES.empty:
    RES[RES.kind == 'design'].to_csv(f'{OUT}/designs.csv', index=False)
    _designs_for_density = pd.read_csv(f'{OUT}/designs.csv')
    _dens = (_designs_for_density.groupby(['spec','arm','target'])
               .agg(n=('run_id','size'), degenerate_frac=('degenerate','mean'),
                    median_mpnn=('mpnn_score','median'), median_len=('binder_len','median'))
               .round(3).reset_index())
    _dens.to_csv(f'{BASE}/gate1/designability.csv', index=False)

# ── 2. design specs (hotspots, contigs, crops) ────────────────────────────────
spec_rows = []
for p in sorted(glob.glob(f'{BASE}/design_specs/*.json')):
    s = jl(p)
    spec_rows.append(dict(
        spec=s.get('name'), id=s.get('id'), pdb_id=s.get('pdb_id'), target=s.get('target'),
        partner=s.get('partner'), arm=s.get('arm'),
        hotspots=' '.join(s.get('hotspots', [])), n_hotspots=len(s.get('hotspots', [])),
        hotspot_precision=s.get('hotspot_precision'),
        n_target_res_after_crop=s.get('n_target_res_after_crop'), crop_radius_A=s.get('crop_radius_A'),
        n_segments=len(s.get('segments', [])), contig=s.get('contig'),
        binder_len_lo=(s.get('binder_len_range') or [None, None])[0],
        binder_len_hi=(s.get('binder_len_range') or [None, None])[1],
        native_partner_len=s.get('native_partner_len'), num_designs=s.get('num_designs')))
SPECS = pd.DataFrame(spec_rows)

# ── 3. gold-standard target metadata ──────────────────────────────────────────
gold_rows = []
for d in sorted(glob.glob(f'{BASE}/gold/*/interface_residues.json')):
    g = jl(d); key = os.path.basename(os.path.dirname(d))
    tseq = g.get('target_seq', {}); pseq = g.get('partner_seq', {})
    hf = [v for v in (g.get('partner_helix_frac') or {}).values() if v is not None]
    gold_rows.append(dict(
        gold_dir=key, pdb_id=g.get('pdb_id'),
        target_chains=','.join(g.get('target_chains', [])),
        partner_chains=','.join(g.get('partner_chains', [])),
        n_target_res=len(g.get('target_residues', [])),
        n_iface_contact_5A=len(g.get('interface_contact_5A', [])),
        n_iface_dsasa=len(g.get('interface_dsasa', [])),
        n_iface_consensus=len(g.get('interface_consensus', [])),
        n_partner_iface=len(g.get('partner_interface_5A', [])),
        partner_len=g.get('partner_len'),
        partner_helix_frac=max(hf) if hf else None,
        has_numbering_gaps=any(g.get('target_gaps', {}).values()),
        flags=';'.join(g.get('flags', [])),
        target_seq=' / '.join(tseq.values()), partner_seq=' / '.join(pseq.values()),
        interface_consensus_residues=' '.join(g.get('interface_consensus', []))))
GOLD = pd.DataFrame(gold_rows)

# ── 4. GNN residue-level benchmark + target quality + designability ───────────
GNN_EVAL = rd(f'{BASE}/results/gnn_v3_benchmark_eval.csv')
TQ = rd(f'{BASE}/gate1/target_quality.csv')
if not TQ.empty: TQ = TQ.rename(columns={TQ.columns[0]: 'target'})
DESIGNABILITY = rd(f'{BASE}/gate1/designability.csv')
P9 = rd(f'{BASE}/results/phase9_per_target_arm.csv')

# ── 5. derived per-target x arm summary (the headline table) ──────────────────
per_ta = pd.DataFrame()
if not RES.empty:
    d = RES[(RES.kind == 'design') & RES.pose_rmsd.notna()].copy()
    # unified with Phase9 cell 7's identical inline formula -- see pipeline_metrics.py
    d['hit'], d['self_consistent'] = classify_design(d.pose_rmsd, d.com_dist)
    per_ta = (d.groupby(['target', 'arm'])
                .agg(n=('run_id', 'size'),
                     hit_rate=('hit', 'mean'),
                     frac_self_consistent=('self_consistent', 'mean'),
                     median_pose_rmsd=('pose_rmsd', 'median'),
                     median_com_dist=('com_dist', 'median'),
                     median_iptm=('iptm', 'median'),
                     median_iface_pae=('iface_pae', 'median'),
                     median_plddt=('plddt', 'median'),
                     best_f1=('f1', 'max'), median_f1=('f1', 'median'),
                     median_mpnn_score=('mpnn_score', 'median'),
                     degenerate_frac=('degenerate', 'mean'),
                     median_rmsd_target_fit=('rmsd_target_fit', 'median'))
                .round(4).reset_index())

# ── 6. per-target roll-up: biology + evaluator quality + every arm's score ────
per_t = pd.DataFrame()
if not per_ta.empty:
    wide = {}
    for metric in ('hit_rate', 'frac_self_consistent', 'best_f1', 'median_iptm'):
        w = per_ta.pivot(index='target', columns='arm', values=metric)
        w.columns = [f'{metric}__{c}' for c in w.columns]; wide[metric] = w
    per_t = pd.concat(wide.values(), axis=1).reset_index()
    if not SPECS.empty:
        hp = SPECS.pivot_table(index='target', columns='arm', values='hotspot_precision')
        hp.columns = [f'hotspot_precision__{c}' for c in hp.columns]
        cs = SPECS.pivot_table(index='target', columns='arm', values='n_target_res_after_crop')
        cs.columns = [f'crop_residues__{c}' for c in cs.columns]
        per_t = per_t.merge(hp.reset_index(), on='target', how='left').merge(cs.reset_index(), on='target', how='left')
        meta = SPECS.groupby('target').agg(pdb_id=('pdb_id', 'first'), partner=('partner', 'first'),
                                           id=('id', 'first')).reset_index()
        per_t = per_t.merge(meta, on='target', how='left')
    if not GOLD.empty:
        G = GOLD.copy(); G['id'] = G.gold_dir.str.split('_').str[0].astype(int)
        per_t = per_t.merge(G[['id', 'n_target_res', 'n_iface_consensus', 'partner_len',
                               'partner_helix_frac', 'has_numbering_gaps']], on='id', how='left')
    if not GNN_EVAL.empty and 'method' in GNN_EVAL.columns:
        ge = GNN_EVAL[GNN_EVAL.method == 'gnn_post'][['target', 'auroc', 'auprc', 'p_spatial']]
        per_t = per_t.merge(ge.rename(columns={'auroc': 'gnn_residue_auroc',
                                               'auprc': 'gnn_residue_auprc',
                                               'p_spatial': 'gnn_spatial_null_p'}), on='target', how='left')
    if not TQ.empty:
        keep = [c for c in ('target', 'f1', 'com_dist', 'iptm', 'target_fit', 'trusted') if c in TQ.columns]
        per_t = per_t.merge(TQ[keep].rename(columns={'f1': 'native_ctrl_f1', 'com_dist': 'native_ctrl_com_dist',
                                                     'iptm': 'native_ctrl_iptm', 'target_fit': 'native_ctrl_target_fit'}),
                            on='target', how='left')

# ── 7. paired statistics across targets ───────────────────────────────────────
stats_rows = []
if not per_ta.empty:
    try:
        from scipy.stats import wilcoxon
        trusted = set(per_t.loc[per_t.get('trusted', True) == True, 'target']) if 'trusted' in per_t else set(per_ta.target)
        for metric in ('hit_rate', 'frac_self_consistent', 'best_f1'):
            piv = per_ta[per_ta.target.isin(trusted)].pivot(index='target', columns='arm', values=metric)
            for a, b in [('gnn', 'baseline'), ('gnn', 'scrambled'), ('oracle', 'gnn'),
                         ('oracle', 'baseline'), ('baseline', 'scrambled')]:
                if a not in piv or b not in piv: continue
                x = piv[[a, b]].dropna()
                if len(x) < 3: continue
                try: p = float(wilcoxon(x[a], x[b], zero_method='wilcox').pvalue)
                except Exception: p = np.nan
                stats_rows.append(dict(metric=metric, arm_a=a, arm_b=b, n_targets=len(x),
                                       wins_a=int((x[a] > x[b]).sum()), wins_b=int((x[b] > x[a]).sum()),
                                       ties=int((x[a] == x[b]).sum()),
                                       median_a=round(float(x[a].median()), 4),
                                       median_b=round(float(x[b].median()), 4),
                                       mean_a=round(float(x[a].mean()), 4),
                                       mean_b=round(float(x[b].mean()), 4),
                                       wilcoxon_p=p))
    except ImportError:
        pass
STATS = pd.DataFrame(stats_rows)

# ── 8. provenance ─────────────────────────────────────────────────────────────
man = {}
for p in sorted(glob.glob(f'{BASE}/manifests/*.json')):
    man[os.path.basename(p)] = jl(p)
MANIFEST = pd.DataFrame([{'manifest': k, 'key': kk, 'value': str(vv)}
                         for k, v in man.items() for kk, vv in (v.items() if isinstance(v, dict) else [])])

README = pd.DataFrame([
 ('designs',        'One row per designed binder (n=1200): sequence, ProteinMPNN score, all Boltz-2 metrics, pose/placement geometry.'),
 ('controls',       'Native-partner (positive) and shuffled-sequence (negative) control runs.'),
 ('per_target_arm', 'Target x arm summary: hit rate, pose self-consistency, confidence, F1. THE headline table.'),
 ('per_target',     'One row per target: biology (interface size, partner length, gaps), GNN residue-level AUROC, hotspot precision and crop size per arm, native-control quality, and every arm score.'),
 ('specs',          'The 80 RFdiffusion design specs: hotspots, contig string, crop size, binder length range.'),
 ('gold_targets',   'Gold-standard structures: chains, interface definitions (5A contact / dSASA / consensus), sequences, numbering gaps.'),
 ('gnn_benchmark',  'Phase 2b residue-level interface prediction: AUROC/AUPRC/R-precision per target per method, with spatial-null p-values.'),
 ('paired_stats',   'Paired Wilcoxon tests across targets for every arm pair and metric.'),
 ('designability',  'ProteinMPNN designability per spec (fraction of backbones yielding non-degenerate sequences).'),
 ('target_quality', 'Native-control validation per target; trusted=False means the evaluator could not reproduce the native complex.'),
 ('manifests',      'Library versions, seeds and counts recorded per phase.'),
], columns=['sheet', 'description'])

# ── 9. write ──────────────────────────────────────────────────────────────────
stamp = time.strftime('%Y%m%d')
xlsx = f'{BASE}/MASTER_PPI_benchmark_{stamp}.xlsx'
sheets = {'README': README, 'designs': RES[RES.kind == 'design'] if not RES.empty else RES,
          'controls': RES[RES.kind != 'design'] if not RES.empty else RES,
          'per_target_arm': per_ta, 'per_target': per_t, 'specs': SPECS, 'gold_targets': GOLD,
          'gnn_benchmark': GNN_EVAL, 'paired_stats': STATS, 'designability': DESIGNABILITY,
          'target_quality': TQ, 'phase9_per_target_arm': P9, 'manifests': MANIFEST}
with pd.ExcelWriter(xlsx, engine='openpyxl') as xw:
    for name, df in sheets.items():
        if df is None or (hasattr(df, 'empty') and df.empty):
            pd.DataFrame({'note': [f'{name}: source file not found']}).to_excel(xw, sheet_name=name[:31], index=False)
        else:
            df.to_excel(xw, sheet_name=name[:31], index=False)
for name, df in sheets.items():
    if df is not None and hasattr(df, 'empty') and not df.empty:
        df.to_csv(f'{OUT}/{name}.csv', index=False)

print(f'MASTER workbook -> {xlsx}  ({os.path.getsize(xlsx)/1e6:.1f} MB)')
print(f'flat CSVs       -> {OUT}/')
for name, df in sheets.items():
    print(f'  {name:<22} {len(df) if hasattr(df, "__len__") else 0:>6} rows')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MASTER workbook -> /content/drive/MyDrive/Protein_interactions/MASTER_PPI_benchmark_20261017.xlsx  (0.4 MB)
flat CSVs       -> /content/drive/MyDrive/Protein_interactions/MASTER/
  README                     11 rows
  designs                  1200 rows
  controls                   80 rows
  per_target_arm             80 rows
  per_target                 20 rows
  specs                      80 rows
  gold_targets               20 rows
  gnn_benchmark             140 rows
  paired_stats               15 rows
  designability               0 rows
  target_quality             20 rows
  phase9_per_target_arm      80 rows
  manifests                  40 rows


In [8]:
import pandas as pd
d = pd.read_csv(f'{BASE}/MASTER/designs.csv')
dens = (d.groupby(['spec','arm','target'])
          .agg(n=('run_id','size'), degenerate_frac=('degenerate','mean'),
               median_mpnn=('mpnn_score','median'), median_len=('binder_len','median'))
          .round(3).reset_index())
dens.to_csv(f'{BASE}/gate1/designability.csv', index=False)
print(dens.groupby('arm').degenerate_frac.mean().round(3).to_dict())

{'baseline': 0.263, 'gnn': 0.19, 'oracle': 0.223, 'scrambled': 0.297}
